# Ideas 9 and 1 of the ideas planner: finer matching level and pose feedback, at the true place
Same oracle set-up as Idea 0 (true place is the only candidate; 12 low-altitude official150 photos ours skips, 8 controls ours answers right).
Facts found first: camera 320 x 256, fx 200 -> photo GSD = height / 200 = 0.5-0.75 m/px at 100-150 m; the map is 0.3 m/px; our lowest level 3 is 0.85 m/px, so the canvas is COARSER than the photo (the renderer already blurs the photo to the canvas sharpness). Idea 9's premise (photo finer than map) is inverted; the real variant is a finer canvas.
Variants at the truth, same pose grid (heading -10..10 step 5 x scale 0.90..1.10, 35 poses): A level 3 (as the pipeline), B level 3 + extra blur 1.5 px on the canvas (idea 9 as written), C level 2 (0.60 m/px, canvas covers 134 m), D level 1 (0.42 m/px, 95 m).
Idea 1 pose feedback: from the best pose of A, one finer grid (heading +-2.5 step 1.25, scale x0.94..1.06 step 0.03) at level 3.
Reading: a variant that lifts most skipped photos to >= 8 inliers at the truth AND keeps controls up is worth a retrieval-wide test; else dead.


In [1]:
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)

DATA = os.environ.get("GATE_DATA", "official")
MAP = "new"
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))   # same pool and setup seed as Test_Iteration_3, so NUDT seed 101 / 202 pick the same photos
WIDE_HEADINGS = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]
WIDE_SCALES = [0.95, 1.0, 1.05]
K_PER_QUERY, M_PER_QUERY = 50, 10
MATCHER_CONFIDENCE, LOFTR_COARSE_THRESHOLD = 0.22, 0.21
MIN_INLIERS, GAMMA = 8, 1.0
NUDT_SEED = int(os.environ.get("NUDT_SEED", "101"))
N_NUDT = int(os.environ.get("N_NUDT", "300"))
EXCLUDE_SEEDS = [int(s) for s in os.environ.get("EXCLUDE_SEEDS", "").split(",") if s.strip()]
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
EXCLUDE_N = int(os.environ.get("EXCLUDE_N", "300"))   # size of the excluded seeds' samples (seed 101 picked 300; this reproduces Test_Iteration_3's seed 202 photos exactly)
MATCHER = os.environ.get("MATCHER", "loftr")          # "loftr" (the pipeline) or "xoftr" (MINIMA XoFTR, a swap-in with the same plumbing)
XOFTR_COARSE, XOFTR_FINE = float(os.environ.get("XOFTR_COARSE", "0.3")), float(os.environ.get("XOFTR_FINE", "0.1"))
FRESH_EXCLUDE_POOL = int(os.environ.get("FRESH_EXCLUDE_POOL", "0"))   # >0 (NUDT, with POOL_PHOTOS >= all frames): skip every photo of the usual
# pool of this size (setup seed 7), so the sample is from photos no earlier test has used
EXCLUDE_IDS_FILE = os.environ.get("EXCLUDE_IDS_FILE")   # optional csv with a photo column: never draw these photos (an earlier fresh sample)
PHOTOS_FILE = os.environ.get("PHOTOS_FILE")           # optional csv with a photo column: only these photos
print(MATCHER, DATA, "seed", NUDT_SEED, "n", N_NUDT, "exclude", EXCLUDE_SEEDS, "limit", N_LIMIT)

loftr official seed 101 n 300 exclude [] limit 0


In [2]:
from train.checks.rich_dump import RichDumpPipeline, pack_photo
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=GAMMA, k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY,
                            matcher="loftr_minima", min_inliers=MIN_INLIERS, ransac_pixels=None, matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP,
                            map_margin_m=None, loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=None,
                            headings=WIDE_HEADINGS, scale_factors=WIDE_SCALES)
start = time.time()
pipeline.setup(POOL_PHOTOS, 7)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set
if MATCHER == "xoftr":
    from train.checks.xoftr_batched import load_xoftr_batched      # the src loader lacks the batch ids the batched LoFTR plumbing reads
    pipeline.matcher = load_xoftr_batched(ROOT / "dataset/NUDT/checkpoint/MINIMA/minima_xoftr.ckpt", min_confidence=XOFTR_FINE,
                                          coarse_threshold=XOFTR_COARSE, fine_threshold=XOFTR_FINE)
    print("matcher: XoFTR (coarse %.2f, fine %.2f)" % (XOFTR_COARSE, XOFTR_FINE))

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 112 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup done in 127.8 s
data: official | 379 photos available | zoom levels [3, 4, 5, 6, 7]
photos: official | map: new
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.22 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
batches: 1 LoFTR pairs and 25 DINO places per call
device: cuda
queries per photo: 27 (headings [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0], scale factors [0.95, 1.0, 1.05]); K and M are per query; 1 queries matched together


In [3]:
import math, gc, torch
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from fine_matching.loftr_matcher import candidate_matches, stage2_from_matches
from image_processing.geometry import render_query
T7 = ROOT / "tests/Test_Iteration_7/results"
dump = replay.load_set(T7 / "rich_dump" / "official150")
LOW = 150.0
low, control = [], []
for pid, photo in dump.items():
    if photo["height"] > LOW:
        continue
    r = replay.decide_current(photo)
    if r["stage"] == "skip":
        low.append(pid)
    elif r["err_sigma"] <= 1.0:
        control.append(pid)
control = control[:8]
print(len(low), "low-altitude photos ours skips;", len(control), "controls")

12 low-altitude photos ours skips; 8 controls


In [4]:
HEADINGS = [-15.0, -10.0, -5.0, 0.0, 5.0, 10.0, 15.0]
SCALES = [0.80, 0.90, 0.95, 1.0, 1.05, 1.10, 1.25]
rows = []
for group, ids in [("skipped", low), ("control", control)]:
    for pid in ids:
        photo = dump[pid]; item = pipeline.test_set.items[pid]; level = photo["level"]
        tx, ty = [float(v) for v in photo["truth"]]
        peaks = np.array([[1.0, tx, ty, 0.0, 0.0]])
        for h in HEADINGS:
            for s in SCALES:
                canvas, valid = render_query(item, pipeline.camera, level, query_geometry(h, s))
                stored = candidate_matches(pipeline.matcher, canvas, valid, level, peaks, pipeline.level_images)
                s2 = stage2_from_matches(stored, level, MATCHER_CONFIDENCE)
                err = float(np.hypot(*s2["shifts"][0])) / photo["sigma"] if s2["inliers"][0] > 0 else np.nan
                rows.append(dict(group=group, photo=pid, height=photo["height"], heading=h, scale=s, inliers=float(s2["inliers"][0]), shift_sigma=err))
        torch.cuda.empty_cache()
O = pd.DataFrame(rows); O.to_csv(T7 / "idea0_oracle.csv", index=False); print(len(O), "runs")

980 runs


In [5]:
import cv2
from header import settings
settings.LEVELS = [1, 2, 3, 4, 5, 6, 7]     # allow finer levels for this test only (nothing in src is changed)
from image_processing.map_levels import build_level_images, normalise_reference, level_gsd
sat = pipeline.satellite_map
t0 = time.time()
pipeline.store["features"] = None; pipeline.map_features = None; gc.collect()   # free memory (the 6 GB cap); matching does not need the coarse features
raw = build_level_images(sat, [2])
for lv in (2,):
    pipeline.level_images[lv] = normalise_reference(raw[lv]); print("level", lv, level_gsd(lv), "m/px", pipeline.level_images[lv].shape)
del raw; gc.collect()
print("fine levels built in %.0f s" % (time.time() - t0))


level 2 0.6 m/px (22634, 23162)
fine levels built in 75 s


In [6]:
HEADINGS = [-10.0, -5.0, 0.0, 5.0, 10.0]
SCALES = [0.90, 0.95, 1.0, 1.05, 1.10]
VARIANTS = [("A_level3", 3, 0.0), ("B_level3_blur1.5", 3, 1.5), ("C_level2", 2, 0.0)]
def run_pose(item, photo, lv, blur, h, s, geometry_extra=None):
    geom = query_geometry(h, s)
    canvas, valid = render_query(item, pipeline.camera, lv, geom)
    if blur > 0:
        canvas = cv2.GaussianBlur(canvas, (0, 0), blur)
    tx, ty = [float(v) for v in photo["truth"]]
    peaks = np.array([[1.0, tx, ty, 0.0, 0.0]])
    stored = candidate_matches(pipeline.matcher, canvas, valid, lv, peaks, pipeline.level_images)
    s2 = stage2_from_matches(stored, lv, MATCHER_CONFIDENCE)
    gsd = level_gsd(lv)
    shift_m = float(np.hypot(*s2["shifts"][0])) if s2["inliers"][0] > 0 else np.nan
    return float(s2["inliers"][0]), shift_m / photo["sigma"] if shift_m == shift_m else np.nan
rows = []
for group, ids in [("skipped", low), ("control", control)]:
    for pid in ids:
        photo = dump[pid]; item = pipeline.test_set.items[pid]
        for name, lv, blur in VARIANTS:
            for h in HEADINGS:
                for s in SCALES:
                    inl, err = run_pose(item, photo, lv, blur, h, s)
                    rows.append(dict(group=group, photo=pid, height=photo["height"], variant=name, heading=h, scale=s, inliers=inl, shift_sigma=err))
        torch.cuda.empty_cache()
    print(group, "done", time.time() - t0)
V = pd.DataFrame(rows); V.to_csv(T7 / "idea9_variants.csv", index=False); print(len(V), "runs")


skipped done 112.40427756309509


control done 137.59796690940857
1500 runs


In [7]:
best = V.loc[V.groupby(["group", "photo", "variant"]).inliers.idxmax()]
tab = best.groupby(["group", "variant"]).agg(mean_best=("inliers", "mean"), median_best=("inliers", "median"), n_ge8=("inliers", lambda x: int((x >= 8).sum())), n=("inliers", "size"),
                                              mean_shift_sigma=("shift_sigma", "mean")).round(2)
tab.to_csv(T7 / "idea9_variants_summary.csv"); print(tab.to_string())
print(best.pivot_table(index=["group", "photo"], columns="variant", values="inliers").to_string())


                          mean_best  median_best  n_ge8   n  mean_shift_sigma
group   variant                                                              
control A_level3              16.00         13.0      6   8              0.56
        B_level3_blur1.5       2.50          2.0      0   8              0.89
        C_level2               6.75          5.0      3   8              0.56
skipped A_level3               4.58          4.5      2  12              0.71
        B_level3_blur1.5       0.00          0.0      0  12               NaN
        C_level2               2.50          0.0      1  12              0.34
variant        A_level3  B_level3_blur1.5  C_level2
group   photo                                      
control 1          43.0               5.0      15.0
        4           7.0               0.0       9.0
        6           9.0               7.0       7.0
        8          16.0               4.0       2.0
        31         11.0               0.0      15.0
        36  

In [8]:
# idea 1 pose feedback: finer grid around the best level-3 pose of each photo
A = V[V.variant == "A_level3"]
rows = []
for (group, pid), g in A.groupby(["group", "photo"]):
    b = g.loc[g.inliers.idxmax()]
    photo = dump[pid]; item = pipeline.test_set.items[pid]
    for dh in (-2.5, -1.25, 0.0, 1.25, 2.5):
        for ds in (0.94, 0.97, 1.0, 1.03, 1.06):
            inl, err = run_pose(item, photo, 3, 0.0, b.heading + dh, b.scale * ds)
            rows.append(dict(group=group, photo=pid, first_best=b.inliers, heading=b.heading + dh, scale=b.scale * ds, inliers=inl, shift_sigma=err))
    torch.cuda.empty_cache()
R = pd.DataFrame(rows); R.to_csv(T7 / "idea1_pose_feedback.csv", index=False)
fb = R.groupby(["group", "photo"]).agg(first_best=("first_best", "first"), refined_best=("inliers", "max")).reset_index()
fb.to_csv(T7 / "idea1_pose_feedback_summary.csv", index=False); print(fb.to_string()); print(fb.groupby("group")[["first_best", "refined_best"]].mean().round(2))
print("skipped reaching >= 8:", (fb[fb.group == "skipped"].refined_best >= 8).sum(), "of", (fb.group == "skipped").sum())


      group  photo  first_best  refined_best
0   control      1        43.0          43.0
1   control      4         7.0           7.0
2   control      6         9.0           9.0
3   control      8        16.0          29.0
4   control     31        11.0          14.0
5   control     36         7.0           7.0
6   control     45        20.0          20.0
7   control     55        15.0          15.0
8   skipped     11         0.0           0.0
9   skipped     35         0.0           0.0
10  skipped     66         5.0           7.0
11  skipped    172        10.0          10.0
12  skipped    176         4.0           4.0
13  skipped    200        14.0          15.0
14  skipped    217         5.0           8.0
15  skipped    266         0.0           0.0
16  skipped    286         5.0          10.0
17  skipped    297         5.0           6.0
18  skipped    306         3.0           6.0
19  skipped    329         4.0           5.0
         first_best  refined_best
group                